# Asistente de consulta de derechos laborales

Responde preguntas de empleados **solo** con la Ley Federal del Trabajo y el artículo 123 constitucional.

Cada respuesta trae:
- La respuesta en lenguaje claro
- Los artículos y fracciones que la respaldan, con el texto citado (cada cita se verifica contra la ley)
- Un aviso explícito cuando la respuesta no está en los documentos.

In [ ]:
import os, sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ)); os.chdir(RAIZ)

import laboral.config
if not os.environ.get("ANTHROPIC_API_KEY"):
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")

## 1. Documentos e índice

In [ ]:
from laboral.descarga import descargar
from laboral import Asistente

descargar()
asistente = Asistente.cargar("v1_articulo")

def preguntar(texto: str):
    r = asistente.preguntar(texto)
    r.mostrar()
    return r

In [ ]:
r = preguntar("¿Me pueden correr por estar embarazada?")

In [ ]:
r = preguntar("¿Cuánto es el salario mínimo este año?")

In [ ]:
r = preguntar("Llevo 4 años en la empresa, ¿cuántos días de vacaciones me tocan?")

In [ ]:
r = preguntar("Me enfermé y el IMSS me dio incapacidad, ¿me pueden despedir y me pagan esos días?")

### Caja de preguntas (opcional, requiere `ipywidgets`)

In [ ]:
try:
    import ipywidgets as w
    from IPython.display import display, clear_output
    caja, boton, salida = w.Textarea(placeholder="Escribe tu pregunta…", layout=w.Layout(width="100%")), w.Button(description="Preguntar"), w.Output()
    def _click(_):
        with salida:
            clear_output(); preguntar(caja.value)
    boton.on_click(_click); display(caja, boton, salida)
except ImportError:
    print("Instala ipywidgets o usa preguntar('...')")

In [ ]:
print("Consultas de búsqueda:")
for c in r.consultas: print("  -", c)
print("\nFragmentos recuperados:")
for f in r.fragmentos: print("  -", f.etiqueta, "|", f.contexto[:60])
print(f"\nEstado del modelo: {r.estado_modelo} -> estado final: {r.estado}")
print("Advertencias:", r.advertencias or "ninguna")
print(f"Costo: US${r.costo_usd:.5f} · tokens {r.uso.tokens_entrada} in / {r.uso.tokens_salida} out · {r.segundos_total:.1f} s")
for d in r.uso.detalle: print("  ", d)